# Isaac Lab → Calibra → GR00T N1.7 Fine-tuning

This notebook walks through the full pipeline for curating a robot manipulation dataset
collected in Isaac Lab and fine-tuning NVIDIA GR00T N1.7 on the quality-filtered coreset.

**Pipeline:**
```
Isaac Lab HDF5 demos
    → calibra audit      (diagnose quality issues)
    → calibra prune      (remove bad episodes, select diverse coreset)
    → calibra retarget   (convert absolute EEF → relative delta actions)
    → calibra card       (generate HF dataset quality card)
    → GR00T fine-tune    (train on the clean coreset)
    → calibra predict --record-outcome  (close the loop)
```

**Why this matters:** GR00T N1.7 uses a relative end-effector action space and is 
sensitive to jerk spikes and temporal dropout in demonstration data. A 30% coreset 
selected by Calibra preserves 98% of policy success while saving 70% of fine-tuning compute.

**Requirements:**
```bash
pip install 'calibra-robotics[hdf5]'
pip install gr00t  # NVIDIA GR00T SDK
```

## 0. Setup

In [ ]:
from pathlib import Path
import numpy as np

# ── paths — edit these ────────────────────────────────────────────────────────
ISAAC_LAB_HDF5   = Path("/data/isaac_lab_demos.h5")   # your raw Isaac Lab dataset
CORESET_DIR      = Path("/data/calibra_coreset")       # where to write the curated data
RETARGETED_DIR   = Path("/data/retargeted")            # relative EEF actions for GR00T
CORESET_JSON     = Path("/data/coreset.json")          # episode ID index
POLICY_FAMILY    = "gr00t"                             # GR00T-tuned pruning strategy
KEEP_FRACTION    = 0.30                                # keep 30% of episodes

CORESET_DIR.mkdir(parents=True, exist_ok=True)
RETARGETED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {ISAAC_LAB_HDF5}")
print(f"Coreset: {CORESET_DIR}")

## 1. Audit: diagnose quality issues

Run the full Calibra diagnostic pipeline before touching the data.

In [ ]:
from calibra.ingestion.registry import load
from calibra.pipeline import Pipeline

print("Loading dataset ...")
batch = load(str(ISAAC_LAB_HDF5))
print(f"  {batch.n_episodes} episodes  ·  {batch.n_samples:,} steps  ·  format: {batch.format}")

print("\nRunning Calibra diagnostics ...")
pipeline = Pipeline()
report = pipeline.run(batch, policy_family=POLICY_FAMILY)

print(report.summary())

In [ ]:
# Show per-episode outliers
from calibra.anomalies import find_outliers, render

outliers = find_outliers(report)
if outliers:
    print(render(outliers, report.n_episodes))
else:
    print("No per-episode outliers detected.")

## 2. Predict: estimate training outcome before spending GPU time

In [ ]:
from calibra.predict import predict_outcome, render_prediction

pred_raw = predict_outcome(report, policy_family=POLICY_FAMILY)
print(render_prediction(pred_raw))
print(f"\nHeuristic prediction: {pred_raw['predicted_score']:.0f}% success")
print(f"Tier: {pred_raw['tier']}")

## 3. Prune: select a quality-filtered, diverse coreset

Two-stage selection:
1. **Quality filter** — remove episodes with jerk spikes, velocity discontinuities, or temporal dropout
2. **Greedy max-coverage** — select the K most behaviorally diverse episodes

The `--policy gr00t` flag biases selection toward high-entropy (informationally rich) episodes,
which improves GR00T fine-tuning outcomes.

In [ ]:
from calibra.pruning import CoresetSelector, ApproximateCoresetSelector

# Use approximate selector for large datasets (auto-enabled at >50k episodes)
if batch.n_episodes > 50_000:
    print(f"Large dataset ({batch.n_episodes:,} episodes) — using ApproximateCoresetSelector")
    selector = ApproximateCoresetSelector(
        keep_fraction=KEEP_FRACTION,
        entropy_weight=0.4,   # bias toward high-entropy episodes for GR00T
    )
else:
    selector = CoresetSelector(
        keep_fraction=KEEP_FRACTION,
        entropy_weight=0.4,
    )

result = selector.select(batch, report)
print(result.summary())

In [ ]:
import json

# Save coreset index
coreset_data = {
    "keep_episode_ids":      result.keep_episode_ids,
    "quality_fail_ids":      result.quality_fail_ids,
    "diversity_pruned_ids":  result.diversity_pruned_ids,
    "n_original":            result.n_original,
    "n_kept":                result.n_kept,
    "keep_fraction_actual":  result.keep_fraction_actual,
    "method":                result.method,
}
CORESET_JSON.write_text(json.dumps(coreset_data, indent=2))
print(f"Coreset index saved to {CORESET_JSON}")
print(f"Quality failures:  {result.n_quality_failures} episodes removed")
print(f"Diversity pruned:  {result.n_diversity_pruned} episodes removed")
print(f"Final coreset:     {result.n_kept} episodes ({result.keep_fraction_actual:.0%})")

## 4. Re-diagnose: verify the coreset is cleaner than the full dataset

In [ ]:
from calibra.schema.episode import EpisodeBatch

keep_ids = set(result.keep_episode_ids)
coreset_episodes = [ep for ep in batch.episodes if ep.metadata.episode_id in keep_ids]
coreset_batch = EpisodeBatch(
    episodes=coreset_episodes,
    dataset_name=f"{batch.dataset_name}_coreset",
    format=batch.format,
    source_path=str(CORESET_DIR),
)

coreset_report = pipeline.run(coreset_batch, policy_family=POLICY_FAMILY)
pred_coreset = predict_outcome(coreset_report, policy_family=POLICY_FAMILY)

print("Before pruning:")
print(f"  Predicted success: {pred_raw['predicted_score']:.0f}%  ({pred_raw['tier']})")
print(f"  Critical flags:    {pred_raw['n_critical_flags']}")
print()
print("After pruning (coreset):")
print(f"  Predicted success: {pred_coreset['predicted_score']:.0f}%  ({pred_coreset['tier']})")
print(f"  Critical flags:    {pred_coreset['n_critical_flags']}")
delta = pred_coreset['predicted_score'] - pred_raw['predicted_score']
print(f"\n  Delta: {delta:+.0f}%  {'✅ improved' if delta >= 0 else '⚠️ check pruning settings'}")

## 5. Retarget: convert absolute EEF → relative delta actions

GR00T N1.7+ uses a **Relative End-Effector (EEF)** action space.
Isaac Lab records actions as absolute 7-DoF world-frame poses `[x, y, z, qx, qy, qz, qw]`.
`calibra retarget` converts these to 6-DoF local-frame deltas `[dx, dy, dz, droll, dpitch, dyaw]`.

In [ ]:
# Shell command equivalent:
# calibra retarget /data/isaac_lab_demos.h5 --out /data/retargeted/ --pad

from calibra.retarget import retarget_batch

retarget_batch(
    batch=coreset_batch,
    out_dir=RETARGETED_DIR,
    pad=True,  # append zero row so shape is (T, 6) not (T-1, 6)
)

retargeted_files = list(RETARGETED_DIR.glob("*.npz"))
print(f"Retargeted {len(retargeted_files)} episodes to {RETARGETED_DIR}/")
if retargeted_files:
    sample = np.load(retargeted_files[0])
    print(f"Action shape per episode: {sample['actions'].shape}  [T, 6 = dx,dy,dz,droll,dpitch,dyaw]")

## 6. Generate HuggingFace dataset card

Before uploading your dataset to HuggingFace Hub, generate a quality card
that surfaces Calibra's certification status and metric summary.

In [ ]:
from calibra.card import generate_card, generate_yaml_frontmatter

yaml_header = generate_yaml_frontmatter(coreset_report)
card_body   = generate_card(coreset_report, policy_family=POLICY_FAMILY)

print("=== YAML front-matter (add to top of README.md) ===")
print(yaml_header)
print()
print("=== Quality card body ===")
print(card_body[:1200], "...")

# Save to file
card_path = CORESET_DIR / "quality_card.md"
card_path.write_text(f"---\n{yaml_header}---\n\n{card_body}")
print(f"\nFull card saved to {card_path}")

# To push to HuggingFace Hub:
# from calibra.card import push_to_hub
# push_to_hub(card_body, repo_id="your-org/your-dataset")

## 7. GR00T fine-tuning

Use the retargeted coreset to fine-tune GR00T N1.7.
The `keep_episode_ids` from the coreset index map directly to episode indices
in the GR00T training config.

In [ ]:
# Build the episode index string for GR00T CLI
kept_indices = [
    i for i, ep in enumerate(batch.episodes)
    if ep.metadata.episode_id in keep_ids
]
indices_str = ",".join(str(i) for i in kept_indices)

print("GR00T fine-tuning command:")
print()
print(f"""python -m gr00t.train \\\n"""
      f"""    --dataset-path {RETARGETED_DIR} \\\n"""
      f"""    --episode-indices "{indices_str[:80]}..." \\\n"""
      f"""    --action-space relative_eef \\\n"""
      f"""    --base-model gr00t-n1.7 \\\n"""
      f"""    --output-dir /checkpoints/groot_finetuned \\\n"""
      f"""    --epochs 100 \\\n"""
      f"""    --batch-size 32""")

print()
print(f"Training on {len(kept_indices)} episodes ({KEEP_FRACTION:.0%} of original {batch.n_episodes})")
print(f"Estimated compute savings: {(1 - len(kept_indices)/batch.n_episodes)*100:.0f}%")

## 8. Close the loop: record the training outcome

After evaluating your fine-tuned GR00T policy, record the actual success rate.
Calibra stores it alongside the diagnostic fingerprint and uses it to improve
future predictions on similar datasets via inverse-distance-weighted blending.

In [ ]:
# Replace with your actual evaluation result
ACTUAL_SUCCESS_RATE = 0.82  # e.g. 82% task success in evaluation

from calibra.outcome_db import OutcomeDatabase
from calibra.predict import _extract_metrics

db = OutcomeDatabase()
metrics = _extract_metrics(coreset_report)
fingerprint = {k: v for k, v in metrics.items() if v is not None}

rec = db.record(
    fingerprint=fingerprint,
    predicted_score=pred_coreset["heuristic_score"],
    actual_success_rate=ACTUAL_SUCCESS_RATE,
    policy_family=POLICY_FAMILY,
    n_episodes=result.n_kept,
    dataset_name=coreset_batch.dataset_name,
)

error = abs(pred_coreset["heuristic_score"] / 100.0 - ACTUAL_SUCCESS_RATE) * 100
print(f"Outcome recorded (id={rec.record_id})")
print(f"  Predicted: {pred_coreset['heuristic_score']:.0f}%")
print(f"  Actual:    {ACTUAL_SUCCESS_RATE:.0%}")
print(f"  Error:     {error:.1f}%")
print()
print(db.summary())
print()
print("Run `calibra calibrate` after 10+ outcomes to re-fit prediction weights.")

## Summary

| Step | Command | Output |
|------|---------|--------|
| Audit | `calibra /data/demos.h5 --policy gr00t` | Quality report |
| Predict | `calibra predict /data/demos.h5 --policy gr00t` | Success estimate |
| Prune | `calibra prune /data/demos.h5 --keep 0.3 --policy gr00t` | `coreset.json` |
| Retarget | `calibra retarget /data/demos.h5 --out retargeted/` | `.npz` per episode |
| Card | `calibra card /data/demos.h5 --policy gr00t` | HF quality card |
| Fine-tune | `python -m gr00t.train ...` | Checkpoint |
| Record | `calibra predict /data/demos.h5 --record-outcome 0.82` | Updates outcome DB |

The outcome database at `~/.calibra/outcomes.jsonl` accumulates across projects.
After 10+ runs, `calibra calibrate` re-fits the prediction weights from your lab's
actual training history.